STEP **1**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# read counts data
expr = pd.read_csv("SRP_counts.tsv", sep="\t", index_col="Gene")
meta = pd.read_csv("metadata.tsv", sep="\t")

assert list(expr.columns) == meta.refinebio_accession_code.tolist()
print("genes x samples:", expr.shape)

genes x samples: (43363, 335)


In [3]:
meta[["region", "dx"]] = meta.refinebio_subject.str.split("_", n=1, expand=True)
meta["donor"] = meta.refinebio_title.str.split("_").str[0]

print(pd.crosstab(meta.dx, meta.region))
print("unique donors:", meta.donor.nunique())

region            ancg  dlpfc  nacc
dx                                 
bipolar disorder    25     28    30
control             28     28    27
major depression    27     30    32
schizophrenia       27     28    25
unique donors: 96


In [4]:
run_to_lib = meta.set_index("refinebio_accession_code").refinebio_title

print("duplicate runs:", run_to_lib.duplicated().sum())

expr_lib = expr.T.groupby(run_to_lib).mean().T

meta_lib = (
    meta.drop_duplicates("refinebio_title")
    .set_index("refinebio_title")
    .loc[expr_lib.columns]
)

print("genes x libraries:", expr_lib.shape)

duplicate runs: 69
genes x libraries: (43363, 266)


In [5]:
print("range before log:", expr_lib.values.min(), expr_lib.values.max())

log_expr = np.log2(expr_lib + 1)

range before log: -0.1583617006285569 197.4761356701856


STEP **2**

In [6]:
gene_variance = log_expr.var(axis=1)
top_5000 = gene_variance.nlargest(5000).index
expr_5000 = log_expr.loc[top_5000].T
print(expr_5000.shape)

(266, 5000)


In [7]:
ancg_samples = meta_lib.index[meta_lib["region"] == "ancg"]

log_expr_ancg = log_expr[ancg_samples]

print(log_expr_ancg.shape)

(43363, 89)


In [8]:
gene_variance_ancg = log_expr_ancg.var(axis=1)

top_5000_ancg = gene_variance_ancg.nlargest(5000).index

expr_5000_ancg = log_expr_ancg.loc[top_5000_ancg].T

print(expr_5000_ancg.shape)

(89, 5000)


In [13]:
expr_AnCg = expr_5000[
    expr_5000.index.str.contains("_AnCg_")
].copy()

meta_AnCg = meta_lib.loc[expr_AnCg.index].copy()

print("AnCg expression matrix shape:", expr_AnCg.shape)

AnCg expression matrix shape: (89, 5000)


In [18]:
# Random Forest
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

meta_AnCg = meta_lib.loc[expr_AnCg.index].copy()

dx = meta_AnCg["dx"].astype(str).str.strip().str.lower()

y_all = dx.map({
    "control": "Control",
    "bipolar disorder": "Disease",
    "schizophrenia": "Disease",
    "major depression": "Disease"
})

valid = y_all.notna()
X_all = log_expr.loc[:, expr_AnCg.index[valid]].T
y_all = y_all.loc[valid]

print("Group counts:")
print(y_all.value_counts())

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.25,
    random_state=42,
    stratify=y_all
)

gene_variance = X_train_all.var(axis=0)
top_genes = gene_variance.nlargest(
    min(5000, X_train_all.shape[1])
).index

X_train = X_train_all[top_genes]
X_test = X_test_all[top_genes]

print("\nTraining samples:", len(X_train))
print("Test samples:", len(X_test))
print("Selected genes:", len(top_genes))

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=3,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("\nTraining accuracy:", round(rf.score(X_train, y_train), 4))
print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, y_pred), 4)
)

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=["Control", "Disease"],
    zero_division=0
))

print("Confusion matrix (rows = actual, columns = predicted):")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=["Control", "Disease"]),
    index=["Actual Control", "Actual Disease"],
    columns=["Predicted Control", "Predicted Disease"]
))

Group counts:
dx
Disease    67
Control    22
Name: count, dtype: int64

Training samples: 66
Test samples: 23
Selected genes: 5000

Training accuracy: 1.0
Test accuracy: 0.7391
Balanced accuracy: 0.5539

Classification report:
              precision    recall  f1-score   support

     Control       0.50      0.17      0.25         6
     Disease       0.76      0.94      0.84        17

    accuracy                           0.74        23
   macro avg       0.63      0.55      0.55        23
weighted avg       0.69      0.74      0.69        23

Confusion matrix (rows = actual, columns = predicted):
                Predicted Control  Predicted Disease
Actual Control                  1                  5
Actual Disease                  1                 16


In [19]:
# Assignment 3

pam_labels = pd.read_csv("a3_labels_pam_ancg.tsv", sep="\t")

pam_labels = pam_labels[["refinebio_title", "g5000_k4"]].copy()
pam_labels = pam_labels.dropna()
pam_labels["g5000_k4"] = pam_labels["g5000_k4"].astype(int)

sample_ids = pam_labels["refinebio_title"].tolist()

X_all = log_expr.loc[:, sample_ids].T
y_all = pam_labels.set_index("refinebio_title").loc[
    X_all.index, "g5000_k4"
]

assert X_all.index.equals(y_all.index)
assert len(X_all) == 89

print("Cluster counts:")
print(y_all.value_counts().sort_index())

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.25,
    random_state=42,
    stratify=y_all
)

gene_variance = X_train_all.var(axis=0)
top_genes = gene_variance.nlargest(
    min(5000, X_train_all.shape[1])
).index

X_train = X_train_all[top_genes]
X_test = X_test_all[top_genes]

rf_clusters = RandomForestClassifier(
    n_estimators=500,
    max_depth=3,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_clusters.fit(X_train, y_train)

y_pred = rf_clusters.predict(X_test)

print("\nTraining accuracy:", round(rf_clusters.score(X_train, y_train), 4))
print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, y_pred), 4)
)

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion matrix:")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=sorted(y_all.unique())),
    index=[f"Actual cluster {c}" for c in sorted(y_all.unique())],
    columns=[f"Predicted cluster {c}" for c in sorted(y_all.unique())]
))

Cluster counts:
g5000_k4
1    34
2    20
3    12
4    23
Name: count, dtype: int64

Training accuracy: 1.0
Test accuracy: 0.8696
Balanced accuracy: 0.8889

Classification report:
              precision    recall  f1-score   support

           1       0.80      0.89      0.84         9
           2       1.00      1.00      1.00         5
           3       1.00      1.00      1.00         3
           4       0.80      0.67      0.73         6

    accuracy                           0.87        23
   macro avg       0.90      0.89      0.89        23
weighted avg       0.87      0.87      0.87        23

Confusion matrix:
                  Predicted cluster 1  Predicted cluster 2  \
Actual cluster 1                    8                    0   
Actual cluster 2                    0                    5   
Actual cluster 3                    0                    0   
Actual cluster 4                    2                    0   

                  Predicted cluster 3  Predicted cluster 

STEP **3**

In [20]:
import os
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier

os.makedirs("results", exist_ok=True)

pam_labels = pd.read_csv("a3_labels_pam_ancg.tsv", sep="\t")
pam_labels = pam_labels.set_index("refinebio_title")

sample_ids = pam_labels.index.tolist()

X = log_expr.loc[:, sample_ids].T

meta = meta_lib.loc[sample_ids]
dx = meta["dx"].astype(str).str.strip().str.lower()

y_group = dx.map({
    "control": 0,
    "bipolar disorder": 1,
    "schizophrenia": 1,
    "major depression": 1
})

y_cluster = (pam_labels["g5000_k4"] == 1).astype(int)

assert len(X) == 89
assert y_group.notna().all()
assert y_cluster.notna().all()
assert X.index.equals(pam_labels.index)

class TopVarianceGenes(BaseEstimator, TransformerMixin):
    def __init__(self, n_genes=5000):
        self.n_genes = n_genes

    def fit(self, X, y=None):
        variances = np.var(X, axis=0)
        self.selected_indices_ = np.argsort(variances)[
            -min(self.n_genes, X.shape[1]):
        ]
        return self

    def transform(self, X):
        return X[:, self.selected_indices_]

gene_counts = [10, 100, 1000, 5000, 10000]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

predictions = pd.DataFrame(index=X.index)
predictions.index.name = "refinebio_title"

for n in gene_counts:
    for target_name, y in [
        ("group", y_group),
        ("cluster", y_cluster)
    ]:
        model = Pipeline([
            ("select", TopVarianceGenes(n_genes=n)),
            ("rf", RandomForestClassifier(
                n_estimators=500,
                max_depth=3,
                min_samples_leaf=5,
                max_features="sqrt",
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            ))
        ])

        predictions[f"{target_name}_pred_g{n}"] = cross_val_predict(
            model,
            X.to_numpy(),
            y.to_numpy().astype(int),
            cv=cv,
            method="predict"
        )

predictions.to_csv(
    "results/a4_predictions_randomforest.tsv",
    sep="\t"
)

print("Saved results/a4_predictions_randomforest.tsv")
print(predictions.head())
print("Shape:", predictions.shape)

Saved results/a4_predictions_randomforest.tsv
                      group_pred_g10  cluster_pred_g10  group_pred_g100  \
refinebio_title                                                           
X1834_AnCg_C_SL31501               1                 0                1   
X1881_AnCg_B_SL7146                0                 0                1   
X2208_AnCg_M_SL7409                1                 1                1   
X2267_AnCg_M_SL7388                1                 1                0   
X2292_AnCg_C_SL6660                0                 1                1   

                      cluster_pred_g100  group_pred_g1000  cluster_pred_g1000  \
refinebio_title                                                                 
X1834_AnCg_C_SL31501                  0                 1                   0   
X1881_AnCg_B_SL7146                   0                 1                   1   
X2208_AnCg_M_SL7409                   1                 1                   1   
X2267_AnCg_M_SL7388    

STEP **4**

In [21]:
import os
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

os.makedirs("results", exist_ok=True)

pam_labels = pd.read_csv("a3_labels_pam_ancg.tsv", sep="\t")
pam_labels = pam_labels.set_index("refinebio_title")

sample_ids = pam_labels.index.tolist()
X = log_expr.loc[:, sample_ids].T
meta = meta_lib.loc[sample_ids]

dx = meta["dx"].astype(str).str.strip().str.lower()
y_group = dx.map({
    "control": 0,
    "bipolar disorder": 1,
    "schizophrenia": 1,
    "major depression": 1
})

y_cluster = (pam_labels["g5000_k4"] == 1).astype(int)

assert len(X) == 89
assert y_group.notna().all()
assert X.index.equals(pam_labels.index)

class TopVarianceGenes(BaseEstimator, TransformerMixin):
    def __init__(self, n_genes=5000):
        self.n_genes = n_genes

    def fit(self, X, y=None):
        variances = np.var(X, axis=0)
        n = min(self.n_genes, X.shape[1])
        self.selected_indices_ = np.argsort(variances)[-n:]
        return self

    def transform(self, X):
        return X[:, self.selected_indices_]

gene_counts = [10, 100, 1000, 5000, 10000]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for n in gene_counts:
    for target_name, y in [
        ("group", y_group),
        ("cluster", y_cluster)
    ]:
        y = y.astype(int)

        model = Pipeline([
            ("select", TopVarianceGenes(n_genes=n)),
            ("rf", RandomForestClassifier(
                n_estimators=500,
                max_depth=3,
                min_samples_leaf=5,
                max_features="sqrt",
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            ))
        ])

        probabilities = cross_val_predict(
            model,
            X.to_numpy(),
            y.to_numpy(),
            cv=cv,
            method="predict_proba"
        )[:, 1]

        auc = roc_auc_score(y, probabilities)

        results.append({
            "target": target_name,
            "genes": n,
            "AUC": auc
        })

auc_results = pd.DataFrame(results)

auc_results.to_csv(
    "results/a4_auc_randomforest.tsv",
    sep="\t",
    index=False
)

print(auc_results.pivot(
    index="genes",
    columns="target",
    values="AUC"
).round(4))

print("\nSaved results/a4_auc_randomforest.tsv")

target  cluster   group
genes                  
10       0.6620  0.5346
100      0.7348  0.5882
1000     0.9257  0.5672
5000     0.9176  0.5550
10000    0.9080  0.5244

Saved results/a4_auc_randomforest.tsv


In [22]:
signature_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=3,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

gene_variance = X.var(axis=0)
top_genes = gene_variance.nlargest(
    min(5000, X.shape[1])
).index

signature_model.fit(X[top_genes], y_group.astype(int))

signature = pd.DataFrame({
    "gene": top_genes,
    "symbol": top_genes,
    "coef": signature_model.feature_importances_,
    "target": "group",
    "method": "randomforest"
})

signature = signature.sort_values("coef", ascending=False)

signature.to_csv(
    "results/a4_signature_randomforest.tsv",
    sep="\t",
    index=False
)

print("Saved results/a4_signature_randomforest.tsv")
print(signature.head())

Saved results/a4_signature_randomforest.tsv
                 gene           symbol      coef target        method
2436  ENSG00000259884  ENSG00000259884  0.006100  group  randomforest
1943  ENSG00000168028  ENSG00000168028  0.005852  group  randomforest
4757  ENSG00000064115  ENSG00000064115  0.005432  group  randomforest
3533  ENSG00000088280  ENSG00000088280  0.005177  group  randomforest
3223  ENSG00000167074  ENSG00000167074  0.004662  group  randomforest


In [23]:
import pandas as pd

pred = pd.read_csv(
    "results/a4_predictions_randomforest.tsv",
    sep="\t",
    index_col="refinebio_title"
)

gene_counts = [10, 100, 1000, 5000, 10000]

# Assignment 1
group_table = pd.DataFrame({
    f"{n} genes": pred[f"group_pred_g{n}"].value_counts()
        .reindex([1, 0], fill_value=0).to_numpy()
    for n in gene_counts
}, index=["Disorder", "Control"])

# Assignment 3
cluster_table = pd.DataFrame({
    f"{n} genes": pred[f"cluster_pred_g{n}"].value_counts()
        .reindex([1, 0], fill_value=0).to_numpy()
    for n in gene_counts
}, index=["Cluster 1", "All other clusters"])

print("Assignment 1 predicted class distributions:")
print(group_table)

print("\nAssignment 3 predicted class distributions:")
print(cluster_table)

Assignment 1 predicted class distributions:
          10 genes  100 genes  1000 genes  5000 genes  10000 genes
Disorder        71         75          78          83           88
Control         18         14          11           6            1

Assignment 3 predicted class distributions:
                    10 genes  100 genes  1000 genes  5000 genes  10000 genes
Cluster 1                 31         33          30          32           30
All other clusters        58         56          59          57           59
